### Install the Ultralytics library (YOLO)

In [ ]:
!pip install ultralytics

### Import libraries for image processing, YOLO model, and visualization

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

### Load the YOLOv11 nano pretrained model

In [ ]:
model = YOLO("yolo11n.pt")

### Run inference on the test video with 0.4 confidence threshold and save the output

In [ ]:
results=model(source="/content/test.mp4", conf=0.4, save=True, verbose=False)

### Iterate over results and print bounding box coordinates for each detection

In [ ]:
for r in results:
    boxes = r.boxes
    for box in boxes:
        print(f"Bounding box coordinates: {box.xyxy.round()}")

### Define ground truth bounding boxes for selected frames (frame index: [x1, y1, x2, y2])

In [ ]:
ground_truth = {
    50:  [[1287, 279, 1631, 893]],
    120: [[756, 234, 1252, 900]],
    200: [[636, 190, 1230, 910]]
}

### Define a function to compute Intersection over Union (IoU) between two bounding boxes

In [ ]:
def iou(box_a, box_b):
    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])

    overlap = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - overlap

    return overlap / (union + 1e-6)

### Extract and store only the frames that have ground truth annotations

In [ ]:
capture = cv2.VideoCapture("/content/test.mp4")
stored_frames = {}
current_frame = 0

while True:
    return_value, frame_data = capture.read()
    if not return_value:
        break
    if current_frame in ground_truth:
        stored_frames[current_frame] = frame_data.copy()
    current_frame += 1

capture.release()

### Extract predicted bounding boxes for frames that have ground truth annotations

In [ ]:
predicted_boxes = {}
for frame_num, result in enumerate(results):
    if frame_num not in ground_truth:
        continue

    detections = []
    for detection in result.boxes:
        coords = detection.xyxy[0]
        x1, y1, x2, y2 = int(coords[0]), int(coords[1]), int(coords[2]), int(coords[3])
        detections.append([x1, y1, x2, y2])
    predicted_boxes[frame_num] = detections

### Draw and display predicted (green) and ground truth (red) boxes on each annotated frame

In [ ]:
def overlay_boxes(image, box_list, box_color):
    for x1, y1, x2, y2 in box_list:
        cv2.rectangle(image, (x1, y1), (x2, y2), box_color, 3)

for frame_num in ground_truth:
    display_image = stored_frames[frame_num].copy()
    overlay_boxes(display_image, predicted_boxes[frame_num], (0, 255, 0))
    overlay_boxes(display_image, ground_truth[frame_num], (255, 0, 0))

    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(display_image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

### Compute and print IoU scores between ground truth and predicted boxes for each annotated frame

In [ ]:
for frame_num in ground_truth:
    print(f"Frame {frame_num}:")
    gt_box = ground_truth[frame_num][0]
    for pred_box in predicted_boxes[frame_num]:
        print(f"IoU: {iou(gt_box, pred_box):.4f}")